In [22]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

df = pd.read_csv('online_retail.csv', encoding='unicode_escape')

print(f"Jumlah baris: {len(df)}")
df.head(5)

Jumlah baris: 541909


,index,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [23]:
df.dropna(axis=0, subset=['InvoiceNo', 'StockCode'], inplace=True)
df['InvoiceNo'] = df['InvoiceNo'].astype('str')
df = df[~df['InvoiceNo'].str.contains('C')]
df = df[~df['StockCode'].str.contains('gift')]
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
df = df[df['Country'] == 'United Kingdom']
df = df[~df['StockCode'].isin(['DOT', 'POST', 'M', 'BANK CHARGES'])]

print(f"Jumlah baris: {len(df)}")
df.head(5)

Jumlah baris: 484043


,index,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [24]:
basket = (df.groupby(['InvoiceNo', 'StockCode'])['Quantity']
          .sum().unstack().reset_index().fillna(0)
          .set_index('InvoiceNo'))

basket.head(5).transpose()

InvoiceNo,536365,536366,536367,536368,536369
StockCode,,,,,
10002,0.0,0.0,0.0,0.0,0.0
10080,0.0,0.0,0.0,0.0,0.0
10120,0.0,0.0,0.0,0.0,0.0
10123C,0.0,0.0,0.0,0.0,0.0
10124A,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...
DCGSSBOY,0.0,0.0,0.0,0.0,0.0
DCGSSGIRL,0.0,0.0,0.0,0.0,0.0
PADS,0.0,0.0,0.0,0.0,0.0


In [25]:
basket_sets = basket.map(lambda x: x > 0)

In [42]:
item_counts = df['Description'].value_counts().reset_index()
item_counts.columns = ['Description', 'jumlah_transaksi']

# produk_target = item_counts[item_counts['Description'] == 'PINK REGENCY TEACUP AND SAUCER']
# print(produk_target)

item1 = '22918'
item2 = '22917'
item3 = '22919'
item4 = '22916'


transaksi_kombinasi = basket_sets[(basket_sets[item3] == 1) & (basket_sets[item4] == 1)]

jumlah_bersamaan = len(transaksi_kombinasi)

total_semua_transaksi = len(basket_sets)
support_kombinasi = jumlah_bersamaan / total_semua_transaksi

# print(f"Jumlah transaksi {item1}, {item2}, {item3}, {item4}: {jumlah_bersamaan} transaksi")
print(f"Jumlah transaksi {item1}, {item2}, {item3}, {item4}: {jumlah_bersamaan} transaksi")
print(f"Nilai Support : {support_kombinasi}")

total_transaksi = len(basket_sets)

print(f"Total Transaksi yang ada : {total_transaksi}")

Jumlah transaksi 22918, 22917, 22919, 22916: 200 transaksi
Nilai Support : 0.011169440411035406
Total Transaksi yang ada : 17906


In [ ]:
# frequent_itemsets = apriori(basket_sets, min_support=0.01, use_colnames=True)
# frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(lambda x: len(x))
# print(f"Jumlah Frequent Itemsets ditemukan: {len(frequent_itemsets)}")

# rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5)
# valid_rules = rules[rules['lift'] > 1.0]
# valid_rules = valid_rules.sort_values('lift', ascending=False).reset_index(drop=True)
# print(f"Jumlah Association Rules valid (Confidence >= 50% & Lift > 1): {len(valid_rules)}\n")

# if not frequent_itemsets.empty:
    
#     total_transaksi = len(basket_sets)

#     L1 = frequent_itemsets[frequent_itemsets['length'] == 1].copy()
#     L1['jumlah_transaksi'] = (L1['support'] * total_transaksi).round()

#     L2 = frequent_itemsets[frequent_itemsets['length'] == 2].copy()
#     L2['jumlah_transaksi'] = (L2['support'] * total_transaksi).round()

#     if not L1.empty:
#         print(L1[['itemsets', 'jumlah_transaksi', 'support']].head(10))
#         print(f"\n... Total ada {len(L1)} produk tunggal yang memenuhi min_support.")
#     else:
#         print("Tidak ada data L1.")

#     if not L2.empty:
#         print(L2[['itemsets', 'jumlah_transaksi', 'support']].head(10))
#         print(f"\n... Total ada {len(L2)} pasangan produk yang memenuhi min_support.")
#     else:
#         print("Tidak ada data L2 yang memenuhi minimum support.")

#     if not valid_rules.empty:
#         pd.set_option('display.max_colwidth', None)
#         valid_rules['frekuensi'] = (valid_rules['support'] * total_transaksi).round()
#         print(valid_rules[['antecedents', 'consequents', 'frekuensi', 'support', 'confidence', 'lift']].head(15))
#         print(f"\n... Total ada {len(valid_rules)} pasangan produk yang memenuhi min_support.")
#     else:
#         print("Tidak ada rules yang lolos batasan confidence & lift.")
        
# else:
#     print("Tidak ada rules yang dapat dibentuk.")


Jumlah Frequent Itemsets ditemukan: 2211
Jumlah Association Rules valid (Confidence >= 50% & Lift > 1): 1231

               itemsets  jumlah_transaksi   support
0    frozenset({10133})             189.0  0.010555
1    frozenset({15036})             495.0  0.027644
2  frozenset({15056BL})             259.0  0.014464
3   frozenset({15056N})             406.0  0.022674
4   frozenset({16161P})             216.0  0.012063
5    frozenset({16237})             276.0  0.015414
6    frozenset({17003})             233.0  0.013012
7    frozenset({20668})             209.0  0.011672
8    frozenset({20675})             275.0  0.015358
9    frozenset({20676})             380.0  0.021222

... Total ada 825 produk tunggal yang memenuhi min_support.
                       itemsets  jumlah_transaksi   support
825  frozenset({85099B, 20685})             189.0  0.010555
826   frozenset({20712, 20711})             282.0  0.015749
827   frozenset({21928, 20711})             196.0  0.010946
828   frozenset({